In [1]:
# Download Pancancer-normalized from Xena database:  

# Make directory
!mkdir -p ./data_csvs/mutations
# Mutation Files
!curl -o ./data_csvs/mutations/mc3.v0.2.8.PUBLIC.maf.gz https://api.gdc.cancer.gov/data/1c8cfe5f-e52d-41ba-94da-f15ea1337efc

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  718M  100  718M    0     0  3273k      0  0:03:44  0:03:44 --:--:-- 3545k


In [2]:
!gunzip ./data_csvs/mutations/mc3.v0.2.8.PUBLIC.maf.gz

In [5]:
import pandas as pd 
import os 
import numpy as np

# read mutation data 
maf = pd.read_csv('../data_csvs/mutations/mc3.v0.2.8.PUBLIC.maf', 
                 delimiter='\t', 
                 low_memory=False)
maf

,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Variant_Classification,Variant_Type,...,ExAC_AF_NFE,ExAC_AF_OTH,ExAC_AF_SAS,GENE_PHENO,FILTER,COSMIC,CENTERS,CONTEXT,DBVS,NCALLERS
0,TACC2,0,.,GRCh37,10,123810032,123810032,+,Missense_Mutation,SNP,...,.,.,.,.,PASS,SITE|p.T38M|c.113C>T|3,MUTECT|RADIA|SOMATICSNIPER|MUSE|VARSCANS,GGACACGCCCG,by1000G,5
1,JAKMIP3,0,.,GRCh37,10,133967449,133967449,+,Silent,SNP,...,.,.,.,.,PASS,NONE,MUTECT|RADIA|SOMATICSNIPER|MUSE|VARSCANS,CTGGACGAGGA,byFrequency,5
2,PANX3,0,.,GRCh37,11,124489539,124489539,+,Missense_Mutation,SNP,...,.,.,.,.,PASS,SITE|p.R296Q|c.887G>A|3,MUTECT|RADIA|SOMATICSNIPER|MUSE|VARSCANS,ATGTCGGTGGG,.,5
3,SPI1,0,.,GRCh37,11,47380512,47380512,+,Missense_Mutation,SNP,...,.,.,.,.,PASS,NONE,RADIA|MUSE,GGCTGGGGACA,.,2
4,NAALAD2,0,.,GRCh37,11,89868837,89868837,+,Missense_Mutation,SNP,...,.,.,.,.,PASS,SITE|p.R65C|c.193C>T|4,MUTECT|RADIA|SOMATICSNIPER|MUSE|VARSCANS,TTCTTCGGTAA,.,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3600958,THOC2,0,.,GRCh37,X,122829875,122829875,+,Missense_Mutation,SNP,...,.,.,.,.,PASS,NONE,RADIA|MUSE|VARSCANS,CCTATTAAAGA,.,3
3600959,SLC25A14,0,.,GRCh37,X,129498779,129498779,+,Intron,SNP,...,.,.,.,.,PASS,NONE,SOMATICSNIPER|RADIA|MUTECT|MUSE|VARSCANS,AAAAAGCCTCT,.,5
3600960,FGF13,0,.,GRCh37,X,137714160,137714160,+,3'Flank,SNP,...,.,.,.,.,PASS,NONE,MUTECT|MUSE,AGTTCCTGTCT,.,2
3600961,XIST,0,.,GRCh37,X,73044379,73044379,+,RNA,SNP,...,.,.,.,.,PASS,NONE,RADIA|MUTECT|MUSE|VARSCANS,TTAAACCCTTG,.,4


In [7]:
# Extract BRCA samples

import os
import pandas as pd

print("=== Filtering BRCA MAF data: Hallmarks genes and matching samples ===")

# 1. Load Hallmarks gene list
hallmarks = pd.read_csv("../data_csvs/rna/metadata/hallmarks_signatures.csv").values.flatten()
hallmarks = hallmarks[~pd.isnull(hallmarks)]
hallmarks = pd.unique(hallmarks)

print(f"Loaded {len(hallmarks):,} Hallmarks genes.")

# 2. Load RNA sample list
rna_samples = pd.read_csv("../data_csvs/rna/hallmarks/BRCA/rna_clean.csv")
target_samples = set(rna_samples["sample"].unique())

print(f"Loaded {len(target_samples):,} RNA samples.")

# 3. Load MAF data
maf["patient_id"] = maf["Tumor_Sample_Barcode"].str[:15]

print(f"Original MAF records: {len(maf):,}")
print(f"Original unique genes: {maf['Hugo_Symbol'].nunique():,}")
print(f"Original patients: {maf['patient_id'].nunique():,}")

# 4. Filter by RNA samples
maf_filtered_patients = maf[maf["patient_id"].isin(target_samples)].copy()

print(f"Records after patient filter: {len(maf_filtered_patients):,}")
print(f"Patients after filter: {maf_filtered_patients['patient_id'].nunique():,}")

# 5. Filter by Hallmarks genes
maf_final = maf_filtered_patients[maf_filtered_patients["Hugo_Symbol"].isin(hallmarks)].copy()

print(f"Records after gene filter: {len(maf_final):,}")
print(f"Genes after filter: {maf_final['Hugo_Symbol'].nunique():,}")
print(f"Patients after gene filter: {maf_final['patient_id'].nunique():,}")

# 6. Create output directory
output_dir = "../data_csvs/mutations/BRCA"
os.makedirs(output_dir, exist_ok=True)

# 7. Save filtered data
output_file = os.path.join(output_dir, "brca_hallmarks_maf_filtered.csv")
maf_final.to_csv(output_file, index=False)

print(f"Filtered data saved to: {output_file}")

# 8. Summary
print("\n=== Summary ===")
print(f"Original records: {len(maf):,}")
print(f"Original genes: {maf['Hugo_Symbol'].nunique():,}")
print(f"Original patients: {maf['patient_id'].nunique():,}")
print(f"RNA samples used for filtering: {len(target_samples):,}")
print(f"Hallmarks genes used for filtering: {len(hallmarks):,}")
print(f"Final records: {len(maf_final):,}")
print(f"Final genes: {maf_final['Hugo_Symbol'].nunique():,}")
print(f"Final patients: {maf_final['patient_id'].nunique():,}")
print(f"Retention rate: {len(maf_final) / len(maf) * 100:.1f}%")

# 9. Top mutated genes (verification)
top_genes = maf_final["Hugo_Symbol"].value_counts().head(10)
print("\nTop 10 most frequently mutated genes:")
for gene, count in top_genes.items():
    print(f"  {gene}: {count:,} mutations")

print("\n✅ Processing complete.")
print(f"The filtered MAF dataset contains {len(maf_final):,} mutation records across {maf_final['patient_id'].nunique():,} patients and {maf_final['Hugo_Symbol'].nunique():,} Hallmarks genes.")


=== Filtering BRCA MAF data: Hallmarks genes and matching samples ===
Loaded 4,241 Hallmarks genes.
Loaded 1,218 RNA samples.
Original MAF records: 3,600,963
Original unique genes: 21,332
Original patients: 10,295
Records after patient filter: 134,947
Patients after filter: 1,024
Records after gene filter: 32,014
Genes after filter: 4,051
Patients after gene filter: 1,021
Filtered data saved to: ../data_csvs/mutations/BRCA/brca_hallmarks_maf_filtered.csv

=== Summary ===
Original records: 3,600,963
Original genes: 21,332
Original patients: 10,295
RNA samples used for filtering: 1,218
Hallmarks genes used for filtering: 4,241
Final records: 32,014
Final genes: 4,051
Final patients: 1,021
Retention rate: 0.9%

Top 10 most frequently mutated genes:
  PIK3CA: 425 mutations
  TP53: 376 mutations
  CDH1: 144 mutations
  MAP3K1: 142 mutations
  GATA3: 140 mutations
  RYR2: 120 mutations
  SYNE1: 118 mutations
  DST: 106 mutations
  DMD: 92 mutations
  SPTA1: 84 mutations

✅ Processing complet

In [10]:
# BRCA - Binary Encoding of Mutations (Hallmarks genes)

import pandas as pd
import numpy as np
import os

print("=== BRCA - Binary Encoding of Mutations (Hallmarks genes) ===")

# Load filtered MAF data
maf_file = "../data_csvs/mutations/BRCA/brca_hallmarks_maf_filtered.csv"
brca_maf = pd.read_csv(maf_file)

# Load target samples and gene list
rna_samples = pd.read_csv("../data_csvs/rna/hallmarks/BRCA/rna_clean.csv")
target_samples = sorted(rna_samples["sample"].unique())

hallmarks = pd.read_csv("../data_csvs/rna/metadata/hallmarks_signatures.csv").values.flatten()
hallmarks = hallmarks[~pd.isnull(hallmarks)]
hallmarks = sorted(pd.unique(hallmarks))

# Create binary mutation matrix
mutation_matrix = (
    brca_maf[["patient_id", "Hugo_Symbol"]]
    .assign(mutation=1.0)
    .pivot_table(
        index="patient_id",
        columns="Hugo_Symbol",
        values="mutation",
        aggfunc="max",
        fill_value=0.0
    )
    .reindex(index=target_samples, columns=hallmarks, fill_value=0.0)
)

# Save matrix
output_file = "../data_csvs/mutations/BRCA/brca_mutation_binary.csv"
mutation_matrix_to_save = mutation_matrix.reset_index()
mutation_matrix_to_save = mutation_matrix_to_save.rename(columns={"patient_id": "sample"})
mutation_matrix_to_save.to_csv(output_file, index=False)

# Display minimal output
print(f"Saved binary mutation matrix to: {output_file}")
print(f"Number of samples: {mutation_matrix_to_save.shape[0]}")

# Show first 100 rows
mutation_matrix_to_save.head(100)


=== BRCA - Binary Encoding of Mutations (Hallmarks genes) ===
Saved binary mutation matrix to: ../data_csvs/mutations/BRCA/brca_mutation_binary.csv
Number of samples: 1218


Hugo_Symbol,sample,A2M,AAAS,AADAT,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TCGA-3C-AAAU-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TCGA-3C-AALI-01,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,TCGA-3C-AALJ-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,TCGA-3C-AALK-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TCGA-4H-AAAK-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-A2-A1G0-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,TCGA-A2-A1G1-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97,TCGA-A2-A1G4-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,TCGA-A2-A1G6-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# BRCA - Impact-Weighted Encoding of Mutations (Hallmarks genes)

import pandas as pd
import numpy as np
import os

print("=== BRCA - Impact-Weighted Encoding of Mutations (Hallmarks genes) ===")

# Load filtered MAF data
maf_file = "../data_csvs/mutations/BRCA/brca_hallmarks_maf_filtered.csv"
brca_maf = pd.read_csv(maf_file)

# Load target samples and gene list
rna_samples = pd.read_csv("../data_csvs/rna/hallmarks/BRCA/rna_clean.csv")
target_samples = sorted(rna_samples["sample"].unique())

hallmarks = pd.read_csv("../data_csvs/rna/metadata/hallmarks_signatures.csv").values.flatten()
hallmarks = hallmarks[~pd.isnull(hallmarks)]
hallmarks = sorted(pd.unique(hallmarks))

# Define Impact weights
impact_weights = {
    "HIGH": 5.0,
    "MODERATE": 3.0,
    "LOW": 1.0,
    "MODIFIER": 0.2
}

# Show IMPACT distribution
impact_distribution = brca_maf["IMPACT"].value_counts()
print("IMPACT category distribution:")
for impact, count in impact_distribution.items():
    weight = impact_weights.get(impact, 1.0)
    print(f"  {impact}: {count:,} ({count/len(brca_maf)*100:.1f}%) -> Weight: {weight}")

# Initialize mutation matrix
mutation_matrix = pd.DataFrame(0.0, index=target_samples, columns=hallmarks)

# Process mutations
for idx, row in brca_maf.iterrows():
    sample_id = row["patient_id"]
    gene = row["Hugo_Symbol"]
    impact = row.get("IMPACT", "MODERATE")
    if sample_id in target_samples and gene in hallmarks:
        weight = impact_weights.get(impact, 3.0)
        mutation_matrix.loc[sample_id, gene] += weight

# Save matrix
output_file = "../data_csvs/mutations/BRCA/brca_mutation_impact_weighted.csv"
mutation_matrix_to_save = mutation_matrix.reset_index().rename(columns={"index": "sample"})
mutation_matrix_to_save.to_csv(output_file, index=False)

print(f"\nSaved weighted mutation matrix to: {output_file}")

# Show first 100 rows
mutation_matrix_to_save.head(100)


=== BRCA - Impact-Weighted Encoding of Mutations (Hallmarks genes) ===
IMPACT category distribution:
  MODERATE: 16,788 (52.4%) -> Weight: 3.0
  LOW: 6,080 (19.0%) -> Weight: 1.0
  HIGH: 4,830 (15.1%) -> Weight: 5.0
  MODIFIER: 4,316 (13.5%) -> Weight: 0.2

Saved weighted mutation matrix to: ../data_csvs/mutations/BRCA/brca_mutation_impact_weighted.csv


,sample,A2M,AAAS,AADAT,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TCGA-3C-AAAU-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TCGA-3C-AALI-01,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,TCGA-3C-AALJ-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,TCGA-3C-AALK-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TCGA-4H-AAAK-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-A2-A1G0-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,TCGA-A2-A1G1-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97,TCGA-A2-A1G4-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,TCGA-A2-A1G6-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
